# Практична робота 6. Прототип прогнозування часу доставки
**SmartLogist · Дідик Богдан Ігорович · група 201ДОФ23**

Прототип оцінює кількість хвилин від моменту запиту до вручення. Дані синтетичні: їх створює наведений нижче генератор, це не реальні доставки. Для запуску в Colab оберіть «Виконати всі». Локально потрібні Python і scikit-learn (`pip install scikit-learn`).

In [1]:
import json, math, random
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error

random.seed(23)


## 1. Тестові дані
Самостійно сформовано 300 умовних записів. Для кожного задано відстань маршруту, швидкість кур’єра, транспорт, залишок часу до готовності, кількість позицій, рівень заторів і погоду. Правильний результат - умовний фактичний час, створений окремою формулою з випадковим шумом. Відстань вважаємо вже обчисленою сервісом маршрутизації за GPS і картою. Сортування за номером імітує хронологію: 240 записів для навчання, 40 для калібрування та останні 20 для тесту. Це лише демонстраційні дані.

In [2]:
transport_codes = {'велосипед': 0, 'авто': 1, 'пішки': 2}
weather_codes = {'ясно': 0, 'дощ': 1}
records = []
for index in range(300):
    transport = random.choice(list(transport_codes))
    weather = random.choice(list(weather_codes))
    distance = round(random.uniform(0.5, 7.0), 1)
    speed = round(random.uniform(*{'велосипед': (10, 18), 'авто': (15, 30), 'пішки': (4, 6)}[transport]), 1)
    ready = random.randint(0, 12)
    items = random.randint(1, 6)
    traffic = random.randint(0, 9)
    # Ціль створюється окремо від моделі; шум імітує невідомі затримки.
    traffic_factor = 1 + traffic * (0.045 if transport == 'авто' else 0.025)
    target = max(2.0, ready + distance / speed * 60 * traffic_factor
                 + (2.5 if weather == 'дощ' else 0) + items * 0.35
                 + random.gauss(0, 2.0))
    records.append({'id': index + 1, 'distance_km': distance, 'speed_kmh': speed,
                    'transport': transport, 'ready_min': ready, 'items': items,
                    'traffic': traffic, 'weather': weather, 'actual_min': round(target, 1)})

def features(r):
    return [r['distance_km'], r['speed_kmh'], r['ready_min'], r['items'], r['traffic'],
            int(r['transport'] == 'велосипед'), int(r['transport'] == 'авто'),
            int(r['transport'] == 'пішки'), int(r['weather'] == 'дощ')]

train, calibration, test = records[:240], records[240:280], records[280:]
print(f'Усього: {len(records)}; навчання: {len(train)}; калібрування: {len(calibration)}; тест: {len(test)}')
print('Приклад входу:', {k: v for k, v in test[0].items() if k not in ('id', 'actual_min')})
print('Очікуваний час у тестовому записі:', test[0]['actual_min'], 'хв')

Усього: 300; навчання: 240; калібрування: 40; тест: 20
Приклад входу: {'distance_km': 3.4, 'speed_kmh': 16.7, 'transport': 'велосипед', 'ready_min': 1, 'items': 5, 'traffic': 3, 'weather': 'ясно'}
Очікуваний час у тестовому записі: 14.5 хв


## 2. Навчання та інтервал
Використано `GradientBoostingRegressor`: ансамбль дерев послідовно уточнює попередній прогноз. Категорії транспорту та погоду перетворено на двійкові ознаки. Інтервал отримано за абсолютними помилками на окремих калібрувальних даних: для рівня 90% взято відповідну порядкову статистику. Це демонстрація, а не гарантована точність для реальних доставок.

In [3]:
model = GradientBoostingRegressor(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=23)
model.fit([features(r) for r in train], [r['actual_min'] for r in train])
cal_predictions = model.predict([features(r) for r in calibration])
errors = sorted(abs(r['actual_min'] - float(pred)) for r, pred in zip(calibration, cal_predictions))
rank = min(len(errors), math.ceil((len(errors) + 1) * 0.90))
radius = errors[rank - 1]

def predict_delivery(r):
    prediction = max(0.0, float(model.predict([features(r)])[0]))
    return {'eta_min': round(prediction, 1),
            'interval_min': [round(max(0.0, prediction - radius), 1), round(prediction + radius, 1)],
            'source': 'prototype_ml'}

print(f'Радіус інтервалу за калібрувальною вибіркою: {radius:.1f} хв')

Радіус інтервалу за калібрувальною вибіркою: 5.2 хв


## 3. Перевірка на нових прикладах
«Очікуваний» - значення з тестового набору, «отриманий» - результат моделі. Для колонки «коректно» умовно прийнято помилку не більше 5 хвилин; це навчальний критерій, а не вимога з ПР-5. MAE обчислено для всіх 20 тестових записів. Покриття показує частку фактичних значень, які потрапили в інтервал, на цій малій вибірці.

In [4]:
results = []
for r in test:
    prediction = predict_delivery(r)
    error = abs(r['actual_min'] - prediction['eta_min'])
    results.append({'id': r['id'], 'distance_km': r['distance_km'], 'transport': r['transport'],
                    'traffic': r['traffic'], 'weather': r['weather'], 'speed_kmh': r['speed_kmh'], 'ready_min': r['ready_min'], 'items': r['items'], 'expected': r['actual_min'],
                    'actual': prediction['eta_min'], 'interval': prediction['interval_min'],
                    'correct': error <= 5.0})
mae = mean_absolute_error([r['actual_min'] for r in test],
                          [max(0.0, float(x)) for x in model.predict([features(r) for r in test])])
coverage = sum(x['interval'][0] <= x['expected'] <= x['interval'][1] for x in results) / len(results)
print('ID | км | км/год | транспорт | готовність хв | позицій | затори | погода | очікувано | прогноз | похибка ≤ 5 хв')
for x in results:
    print(f"{x['id']} | {x['distance_km']} | {x['speed_kmh']} | {x['transport']} | {x['ready_min']} | {x['items']} | {x['traffic']} | {x['weather']} | {x['expected']:.1f} | {x['actual']:.1f} | {'так' if x['correct'] else 'ні'}")
print(f'MAE на 20 записах: {mae:.2f} хв')
print(f'Похибка не більше 5 хв: {sum(x["correct"] for x in results)}/{len(results)}')
print(f'Покриття інтервалу: {coverage:.0%}')
summary = {'results': results, 'mae': round(float(mae), 2), 'coverage': coverage,
           'radius': round(radius, 2), 'example_input': {k:v for k,v in test[0].items() if k not in ('id','actual_min')},
           'example_output': predict_delivery(test[0])}
print('Приклад відповіді AI Core:', json.dumps(summary['example_output'], ensure_ascii=False))

ID | км | км/год | транспорт | готовність хв | позицій | затори | погода | очікувано | прогноз | похибка ≤ 5 хв
281 | 3.4 | 16.7 | велосипед | 1 | 5 | 3 | ясно | 14.5 | 19.5 | так
282 | 1.8 | 21.8 | авто | 9 | 3 | 3 | ясно | 15.9 | 15.6 | так
283 | 2.7 | 10.4 | велосипед | 0 | 1 | 6 | ясно | 17.7 | 25.4 | ні
284 | 1.6 | 25.6 | авто | 4 | 2 | 6 | ясно | 12.1 | 11.1 | так
285 | 3.7 | 10.8 | велосипед | 12 | 6 | 9 | ясно | 36.6 | 35.6 | так
286 | 1.1 | 27.9 | авто | 12 | 3 | 1 | ясно | 16.8 | 17.4 | так
287 | 1.1 | 5.3 | пішки | 3 | 3 | 4 | дощ | 20.6 | 18.0 | так
288 | 0.9 | 17.8 | велосипед | 9 | 1 | 0 | ясно | 12.7 | 11.2 | так
289 | 1.3 | 5.0 | пішки | 6 | 2 | 9 | дощ | 24.8 | 32.8 | ні
290 | 3.3 | 17.7 | велосипед | 8 | 3 | 2 | дощ | 21.8 | 23.0 | так
291 | 2.0 | 4.0 | пішки | 3 | 6 | 8 | ясно | 42.0 | 28.7 | ні
292 | 4.2 | 5.7 | пішки | 7 | 1 | 3 | дощ | 58.8 | 57.9 | так
293 | 6.7 | 5.6 | пішки | 4 | 3 | 1 | дощ | 83.1 | 85.9 | так
294 | 1.0 | 27.5 | авто | 12 | 2 | 7 | ясно | 12.3

## 4. Інтеграція
OMS + Kitchen & Partner Service + GPS / карти → Telemetry & Routing Service (перевірка й підготовка ознак) → AI / ML Intelligence Core (цей прототип) → прогноз та інтервал → Telemetry & Routing Service → ETA клієнту; за потреби перевірка об’їзду DP-08 / диспетчер. Прототип повертає Python-словник, який можна серіалізувати в JSON; реальний API, перевірка свіжості GPS, пошук об’їзду і резервний режим тут не реалізовані.

Модель навчена на штучній формулі. Для реальної оцінки потрібні історичні дані, розділені за часом, перевірка свіжості ознак і калібрування інтервалів.